# RQ4: Limitations

**Research Question**: What are the causes of unsuccessful generalization attempts?

This notebook analyzes the limitations of test generalization by examining filtering causes and processing failures:
- **Exclusion Analysis**: Overall filtering of tests, assertions, and generalizations by variant
- **Filter Effectiveness**: Detailed breakdown of which filters cause exclusions
- **SPF Failures**: Symbolic PathFinder execution error categorization
- **Test Failures**: Runtime test execution failures by variant
- **Pipeline Analysis**: Processing stage failures in extended dataset evaluation

In [1]:
from teralizer.config import db_config
from teralizer.rq4_limitations import (
    get_exclusions_summary_data,
    get_filtering_exclusions_data,
    get_spf_failures_data,
    get_test_executions_by_variant_data,
    get_test_failures_by_projects_data,
    compute_exclusion_percentages,
    compute_exclusion_breakdown_filtering_vs_failures,
    compute_filtering_exclusions_summary,
    compute_spf_error_categorization,
    compute_test_failures_by_variant_summary,
    compute_test_failures_by_projects_summary,
    generate_exclusions_summary_table,
    generate_exclusions_breakdown_table,
    generate_filtering_results_table,
    generate_spf_failures_table,
    generate_test_failures_by_variant_table,
    generate_test_failures_by_projects_table,
    generate_exclusions_summary_csv,
    generate_exclusions_breakdown_csv,
    generate_filtering_results_csv,
    generate_spf_failures_csv,
    generate_test_failures_by_variant_csv,
    generate_test_failures_by_projects_csv,
)
from teralizer.exports import save_latex_table, save_csv_data
from teralizer.plotting import setup_paper_style
from IPython.display import display

import pandas as pd

# Database connections
conn_dev = db_config.get_dev_engine()  # Main evaluation dataset
conn_test = db_config.get_test_engine()  # Extended dataset

# Configure paper style
setup_paper_style()

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## Overall Exclusions Summary

Analysis of included and excluded counts by variant and level (Test/Assertion/Generalization).

In [2]:
# Get overall exclusions data
exclusions_raw = get_exclusions_summary_data(conn_dev)
exclusions_summary = compute_exclusion_percentages(exclusions_raw)

print("Overall exclusions by variant and level:")
display(
    exclusions_summary[
        [
            "variant",
            "Type",
            "Total",
            "included_count",
            "excluded_count",
            "included_pct",
            "excluded_pct",
        ]
    ]
)

# Generate LaTeX table
exclusions_table = generate_exclusions_summary_table(exclusions_summary)
print("\n=== LaTeX Table ===")
print(exclusions_table)

# Save LaTeX table
save_latex_table(exclusions_table, "tab-exclusions-summary")

# Generate and save CSV data
exclusions_csv = generate_exclusions_summary_csv(exclusions_summary)
csv_path = save_csv_data(
    exclusions_csv,
    "exclusions-summary-data",
    "Included and excluded counts by variant and level (Test/Assertion/Generalization)",
)

print(f"\nExclusions summary data exported to: {csv_path}")
print(f"Shape: {exclusions_csv.shape}")
print("Sample data:")
display(exclusions_csv.head())

Overall exclusions by variant and level:


,variant,Type,Total,included_count,excluded_count,included_pct,excluded_pct
0,SHARED,Test,23246,19306,3940,83.1,16.9
1,SHARED,Assertion,28923,13836,15087,47.8,52.2
2,BASELINE,Generalization,13836,13814,22,99.8,0.2
3,NAIVE_10_TRIES,Generalization,13836,10743,3093,77.6,22.4
4,NAIVE_50_TRIES,Generalization,13836,9964,3872,72.0,28.0
5,NAIVE_200_TRIES,Generalization,13836,9881,3955,71.4,28.6
6,IMPROVED_10_TRIES,Generalization,13836,11788,2048,85.2,14.8
7,IMPROVED_50_TRIES,Generalization,13836,11660,2176,84.3,15.7
8,IMPROVED_200_TRIES,Generalization,13836,11597,2239,83.8,16.2



=== LaTeX Table ===
\begin{table}[H]
  \caption{Included and excluded counts by variant and level.}
  \label{tab:exclusions-summary}
  \begin{tabular}{llrrr}
    \toprule
    Variant & Type & Total & \multicolumn{1}{c}{Included} & \multicolumn{1}{c}{Excluded} \\
    \midrule
    \VariantShared{} & Test & 23,246 & 19,306\; (83.1\%) & 3,940\; (16.9\%) \\
    \midrule
    \VariantShared{} & Assertion & 28,923 & 13,836\; (47.8\%) & 15,087\; (52.2\%) \\
    \midrule
    \VariantBaseline{} & Generalization & 13,836 & 13,814\; (99.8\%) & 22\; (\phantom{0}0.2\%) \\
    \VariantNaiveA{} & Generalization & 13,836 & 10,743\; (77.6\%) & 3,093\; (22.4\%) \\
    \VariantNaiveB{} & Generalization & 13,836 & 9,964\; (72.0\%) & 3,872\; (28.0\%) \\
    \VariantNaiveC{} & Generalization & 13,836 & 9,881\; (71.4\%) & 3,955\; (28.6\%) \\
    \VariantImprovedA{} & Generalization & 13,836 & 11,788\; (85.2\%) & 2,048\; (14.8\%) \\
    \VariantImprovedB{} & Generalization & 13,836 & 11,660\; (84.3\%) & 2,176\

,variant,exclusion_level,total_count,included_count,excluded_count,included_percentage,excluded_percentage
0,\VariantShared{},Test,23246,19306,3940,83.1,16.9
1,\VariantShared{},Assertion,28923,13836,15087,47.8,52.2
2,\VariantBaseline{},Generalization,13836,13814,22,99.8,0.2
3,\VariantNaiveA{},Generalization,13836,10743,3093,77.6,22.4
4,\VariantNaiveB{},Generalization,13836,9964,3872,72.0,28.0


## Exclusions Breakdown: Filtering vs Failures

Analysis of exclusions split into proactive filtering (TestFilteringTask) and reactive failures (JpfExecutionTask, TestAnalysisTask, TestGeneralizationTask).

In [3]:
# Compute exclusion breakdown (main dataset)
exclusions_breakdown = compute_exclusion_breakdown_filtering_vs_failures(
    conn_dev, exclusions_raw.copy()
)

print("Exclusion breakdown by variant and level (Filtering vs Failures):")
display(
    exclusions_breakdown[
        [
            "variant",
            "Type",
            "Total",
            "included_count",
            "filtering_count",
            "failures_count",
            "included_pct",
            "filtering_pct",
            "failures_pct",
        ]
    ]
)

# Generate LaTeX table for main dataset
exclusions_breakdown_table = generate_exclusions_breakdown_table(
    exclusions_breakdown,
    label="tab:exclusions-breakdown",
    caption="Exclusion breakdown showing proactive filtering and reactive failures.",
)
print("\n=== LaTeX Table (Main Dataset) ===")
print(exclusions_breakdown_table)

# Save LaTeX table
save_latex_table(exclusions_breakdown_table, "tab-exclusions-breakdown")

# Generate and save CSV data
exclusions_breakdown_csv = generate_exclusions_breakdown_csv(exclusions_breakdown)
csv_path = save_csv_data(
    exclusions_breakdown_csv,
    "exclusions-breakdown-data",
    "Exclusion breakdown showing filtering vs failures by variant and level",
)

print(f"\nExclusions breakdown data exported to: {csv_path}")
print(f"Shape: {exclusions_breakdown_csv.shape}")
print("Sample data:")
display(exclusions_breakdown_csv.head())

# Try to get extended dataset and generate table
try:
    exclusions_raw_extended = get_exclusions_summary_data(conn_test)
    exclusions_breakdown_extended = compute_exclusion_breakdown_filtering_vs_failures(
        conn_test, exclusions_raw_extended.copy()
    )

    print("\n\nExtended dataset exclusion breakdown:")
    display(
        exclusions_breakdown_extended[
            [
                "variant",
                "Type",
                "Total",
                "included_count",
                "filtering_count",
                "failures_count",
            ]
        ]
    )

    # Generate LaTeX table for extended dataset
    exclusions_breakdown_extended_table = generate_exclusions_breakdown_table(
        exclusions_breakdown_extended,
        label="tab:exclusions-breakdown-extended",
        caption="Exclusion breakdown showing proactive filtering and reactive failures (extended dataset).",
    )

    print("\n=== LaTeX Table (Extended Dataset) ===")
    print(exclusions_breakdown_extended_table)

    # Save extended dataset table
    save_latex_table(
        exclusions_breakdown_extended_table, "tab-exclusions-breakdown-extended"
    )

    # Export CSV data for extended dataset
    exclusions_breakdown_extended_csv = generate_exclusions_breakdown_csv(
        exclusions_breakdown_extended
    )
    csv_path = save_csv_data(
        exclusions_breakdown_extended_csv,
        "exclusions-breakdown-extended-data",
        "Extended dataset exclusion breakdown",
    )
    print(f"Extended breakdown data exported to: {csv_path}")

except Exception as e:
    print(f"\nExtended dataset exclusion breakdown not available: {e}")

Exclusion breakdown by variant and level (Filtering vs Failures):


,variant,Type,Total,included_count,filtering_count,failures_count,included_pct,filtering_pct,failures_pct
0,SHARED,Test,23246,19306,3933,7,83.1,16.9,0.0
1,SHARED,Assertion,28923,13836,12092,2995,47.8,41.8,10.4
2,BASELINE,Generalization,13836,13814,22,0,99.8,0.2,0.0
3,IMPROVED_10_TRIES,Generalization,13836,11788,2016,32,85.2,14.6,0.2
4,IMPROVED_200_TRIES,Generalization,13836,11597,2207,32,83.8,16.0,0.2
5,IMPROVED_50_TRIES,Generalization,13836,11660,2144,32,84.3,15.5,0.2
6,NAIVE_10_TRIES,Generalization,13836,10743,3061,32,77.6,22.1,0.2
7,NAIVE_200_TRIES,Generalization,13836,9881,3923,32,71.4,28.4,0.2
8,NAIVE_50_TRIES,Generalization,13836,9964,3840,32,72.0,27.8,0.2



=== LaTeX Table (Main Dataset) ===
\begin{table}[H]
  \caption{Exclusion breakdown showing proactive filtering and reactive failures.}
  \label{tab:exclusions-breakdown}
  \begin{tabular}{llrrrr}
    \toprule
    & & & & \multicolumn{2}{r}{Excluded} \\
    \cmidrule(lr){5-6}
    Variant & Type & Total & Included & Filtering & Failures \\
    \midrule
    \VariantShared{} & Test & 23,246 & \phantom{}19,306\; (83.1\%) & \phantom{0,0}3,933\; (16.9\%) & \phantom{0,0,0}7\; (\phantom{0}0.0\%) \\
    \midrule
    \VariantShared{} & Assertion & 28,923 & \phantom{}13,836\; (47.8\%) & \phantom{}12,092\; (41.8\%) & \phantom{}2,995\; (10.4\%) \\
    \midrule
    \VariantBaseline{} & Generalization & 13,836 & \phantom{}13,814\; (99.8\%) & \phantom{0,0,0}22\; (\phantom{0}0.2\%) & \phantom{0,0,0}0\; (\phantom{0}0.0\%) \\
    \VariantImprovedA{} & Generalization & 13,836 & \phantom{}11,788\; (85.2\%) & \phantom{0,0}2,016\; (14.6\%) & \phantom{0,}32\; (\phantom{0}0.2\%) \\
    \VariantImprovedC{} & Ge

,variant,exclusion_level,total_count,included_count,filtering_count,failures_count,included_percentage,filtering_percentage,failures_percentage
0,\VariantShared{},Test,23246,19306,3933,7,83.1,16.9,0.0
1,\VariantShared{},Assertion,28923,13836,12092,2995,47.8,41.8,10.4
2,\VariantBaseline{},Generalization,13836,13814,22,0,99.8,0.2,0.0
3,\VariantImprovedA{},Generalization,13836,11788,2016,32,85.2,14.6,0.2
4,\VariantImprovedC{},Generalization,13836,11597,2207,32,83.8,16.0,0.2




Extended dataset exclusion breakdown:


,variant,Type,Total,included_count,filtering_count,failures_count
0,SHARED,Test,81810,33385,40583,7842
1,SHARED,Assertion,122153,711,121060,382
2,IMPROVED_200_TRIES,Generalization,239,206,23,10



=== LaTeX Table (Extended Dataset) ===
\begin{table}[H]
  \caption{Exclusion breakdown showing proactive filtering and reactive failures (extended dataset).}
  \label{tab:exclusions-breakdown-extended}
  \begin{tabular}{llrrrr}
    \toprule
    & & & & \multicolumn{2}{r}{Excluded} \\
    \cmidrule(lr){5-6}
    Variant & Type & Total & Included & Filtering & Failures \\
    \midrule
    \VariantShared{} & Test & 81,810 & \phantom{}33,385\; (40.8\%) & \phantom{0,0}40,583\; (49.6\%) & \phantom{}7,842\; (\phantom{0}9.6\%) \\
    \midrule
    \VariantShared{} & Assertion & 122,153 & \phantom{0,}711\; (\phantom{0}0.6\%) & \phantom{}121,060\; (99.1\%) & \phantom{0,0}382\; (\phantom{0}0.3\%) \\
    \midrule
    \VariantImprovedC{} & Generalization & 239 & \phantom{0,}206\; (86.2\%) & \phantom{0,0,}23\; (\phantom{0}9.6\%) & \phantom{0,}10\; (\phantom{0}4.2\%) \\
    \bottomrule
  \end{tabular}
\end{table}
Saved LaTeX table to /app/analysis/output/verify/tables/tab-exclusions-breakdown-extended

## Filtering-Based Exclusions

Analysis of filtering results for tests, assertions, and generalizations by filter type and variant.

In [4]:
# Get filtering exclusions data from main dataset
filtering_raw_main = get_filtering_exclusions_data(conn_dev)
filtering_main = compute_filtering_exclusions_summary(filtering_raw_main.copy())

print("Main dataset filtering results:")
display(
    filtering_main[
        [
            "variant",
            "Type",
            "filter_name",
            "total",
            "accept",
            "reject",
            "accept_pct",
            "reject_pct",
        ]
    ]
)

# Generate LaTeX table for main dataset
main_filtering_table = generate_filtering_results_table(
    filtering_main,
    "tab:exclusions-filtering",
    "Filtering results for tests, assertions, and generalizations by filter and (generalization) variant.",
)

print("\n=== Main Dataset Filtering Results ===\n")
print(main_filtering_table)

# Save main dataset table
save_latex_table(main_filtering_table, "tab-exclusions-filtering")

# Export CSV data for main dataset
main_filtering_csv = generate_filtering_results_csv(filtering_main, "main")
csv_path = save_csv_data(
    main_filtering_csv,
    "exclusions-filtering-data",
    "Filtering results for tests, assertions, and generalizations by filter and variant",
)
print(f"Main filtering data exported to: {csv_path}")

Main dataset filtering results:


,variant,Type,filter_name,total,accept,reject,accept_pct,reject_pct
0,SHARED,Test,NoAssertions,21532,19306,2226,89.7,10.3
1,SHARED,Test,NonPassingTest,23246,21719,1527,93.4,6.6
2,SHARED,Test,TestType,23246,23066,180,99.2,0.8
3,SHARED,Assertion,AssertionType,28923,28180,743,97.4,2.6
4,SHARED,Assertion,ExcludedTest,28923,27326,1597,94.5,5.5
5,SHARED,Assertion,MissingValue,28923,21766,7157,75.3,24.7
6,SHARED,Assertion,ParameterType,28923,17835,4458,61.7,15.4
7,SHARED,Assertion,VoidReturnType,28923,21763,3,75.2,0.0
8,BASELINE,Generalization,NonPassingTest,13836,13814,22,99.8,0.2
9,IMPROVED_10_TRIES,Generalization,NonPassingTest,13804,11788,2016,85.4,14.6



=== Main Dataset Filtering Results ===

\begin{table}[H]
  \caption{Filtering results for tests, assertions, and generalizations by filter and (generalization) variant.}
  \label{tab:exclusions-filtering}
  \begin{tabular}{lllrrrr}
    \toprule
    Variant & Type & Filter Name & Total & \multicolumn{1}{c}{Accept} & \multicolumn{1}{c}{Defer} & \multicolumn{1}{c}{Reject} \\
    \midrule
    \VariantShared{} & Test & NoAssertions & 21,532 & 19,306\; (89.7\%) & - & 2,226\; (10.3\%) \\
    \VariantShared{} & Test & NonPassingTest & 23,246 & 21,719\; (93.4\%) & - & 1,527\; (\phantom{0}6.6\%) \\
    \VariantShared{} & Test & TestType & 23,246 & 23,066\; (99.2\%) & - & 180\; (\phantom{0}0.8\%) \\
    \midrule
    \VariantShared{} & Assertion & AssertionType & 28,923 & 28,180\; (97.4\%) & - & 743\; (\phantom{0}2.6\%) \\
    \VariantShared{} & Assertion & ExcludedTest & 28,923 & 27,326\; (94.5\%) & - & 1,597\; (\phantom{0}5.5\%) \\
    \VariantShared{} & Assertion & MissingValue & 28,923 & 21,7

In [5]:
# Try to get extended dataset filtering results
try:
    filtering_raw_extended = get_filtering_exclusions_data(conn_test)
    filtering_extended = compute_filtering_exclusions_summary(
        filtering_raw_extended.copy()
    )

    print("Extended dataset filtering results:")
    display(
        filtering_extended[
            [
                "variant",
                "Type",
                "filter_name",
                "total",
                "accept",
                "reject",
                "accept_pct",
                "reject_pct",
            ]
        ]
    )

    # Generate LaTeX table for extended dataset
    extended_filtering_table = generate_filtering_results_table(
        filtering_extended,
        "tab:exclusions-filtering-extended",
        "Filtering results of the extended dataset for tests, assertions, and generalizations.",
    )

    print("\n=== Extended Dataset Filtering Results ===\n")
    print(extended_filtering_table)

    # Save extended dataset table
    save_latex_table(extended_filtering_table, "tab-exclusions-filtering-extended")

    # Export CSV data for extended dataset
    extended_filtering_csv = generate_filtering_results_csv(
        filtering_extended, "extended"
    )
    csv_path = save_csv_data(
        extended_filtering_csv,
        "exclusions-filtering-extended-data",
        "Extended dataset filtering results",
    )
    print(f"Extended filtering data exported to: {csv_path}")

except Exception as e:
    print(f"Extended dataset filtering not available: {e}")

Extended dataset filtering results:


,variant,Type,filter_name,total,accept,reject,accept_pct,reject_pct
0,SHARED,Test,NoAssertions,56844,33385,23459,58.7,41.3
1,SHARED,Test,NonPassingTest,74308,65567,8741,88.2,11.8
2,SHARED,Test,TestType,74308,65031,9277,87.5,12.5
3,SHARED,Assertion,AssertionType,122153,92986,29167,76.1,23.9
4,SHARED,Assertion,ExcludedTest,122153,101513,20640,83.1,16.9
5,SHARED,Assertion,MissingValue,122153,51425,70728,42.1,57.9
6,SHARED,Assertion,ParameterType,122153,5393,60283,4.4,49.4
7,SHARED,Assertion,ReturnType,122153,11645,39780,9.5,32.6
8,IMPROVED_200_TRIES,Generalization,NonPassingTest,229,206,23,90.0,10.0



=== Extended Dataset Filtering Results ===

\begin{table}[H]
  \caption{Filtering results of the extended dataset for tests, assertions, and generalizations.}
  \label{tab:exclusions-filtering-extended}
  \begin{tabular}{lllrrrr}
    \toprule
    Variant & Type & Filter Name & Total & \multicolumn{1}{c}{Accept} & \multicolumn{1}{c}{Defer} & \multicolumn{1}{c}{Reject} \\
    \midrule
    \VariantShared{} & Test & NoAssertions & 56,844 & 33,385\; (58.7\%) & - & 23,459\; (41.3\%) \\
    \VariantShared{} & Test & NonPassingTest & 74,308 & 65,567\; (88.2\%) & - & 8,741\; (11.8\%) \\
    \VariantShared{} & Test & TestType & 74,308 & 65,031\; (87.5\%) & - & 9,277\; (12.5\%) \\
    \midrule
    \VariantShared{} & Assertion & AssertionType & 122,153 & 92,986\; (76.1\%) & - & 29,167\; (23.9\%) \\
    \VariantShared{} & Assertion & ExcludedTest & 122,153 & 101,513\; (83.1\%) & - & 20,640\; (16.9\%) \\
    \VariantShared{} & Assertion & MissingValue & 122,153 & 51,425\; (42.1\%) & - & 70,728\; (5

## SPF Execution Failures

Analysis of Symbolic PathFinder execution failures by error type.

In [6]:
# Get SPF failures data
spf_raw = get_spf_failures_data(conn_dev)
spf_categorized = compute_spf_error_categorization(spf_raw)

print("SPF execution failures by error type:")
display(spf_categorized)

# Generate LaTeX table
spf_table = generate_spf_failures_table(spf_categorized)
print("\n=== SPF Failures Table ===\n")
print(spf_table)

# Save LaTeX table
save_latex_table(spf_table, "tab-exclusions-spf")

# Generate and save CSV data
spf_csv = generate_spf_failures_csv(spf_categorized)
csv_path = save_csv_data(
    spf_csv, "exclusions-spf-data", "SPF execution failures by error type"
)

print(f"\nSPF errors data exported to: {csv_path}")
print(f"Shape: {spf_csv.shape}")
print("Sample data:")
display(spf_csv)

SPF execution failures by error type:


,Error Type,Total,Percent
0,SPF exception,1540,51.42
1,PC size limit exceeded,790,26.38
2,Depth limit exceeded,524,17.50
3,Teralizer exception,97,3.24
4,Execution timeout,28,0.93
5,OutOfMemoryError,16,0.53



=== SPF Failures Table ===

\begin{table}[H]
  \caption{Number of SPF execution failures by error type.}
  \label{tab:exclusions-spf}
  \begin{tabular}{lrr}
    \toprule
    Error Type & Total & Percent \\
    \midrule
    SPF exception & 1,540 & 51.42 \\
    PC size limit exceeded & 790 & 26.38 \\
    Depth limit exceeded & 524 & 17.50 \\
    Teralizer exception & 97 & 3.24 \\
    Execution timeout & 28 & 0.93 \\
    OutOfMemoryError & 16 & 0.53 \\
    \bottomrule
  \end{tabular}
\end{table}
Saved LaTeX table to /app/analysis/output/verify/tables/tab-exclusions-spf.tex

SPF errors data exported to: /app/analysis/output/verify/data/exclusions-spf-data.csv
Shape: (6, 3)
Sample data:


,error_type,failure_count,percentage
0,SPF exception,1540,51.42
1,PC size limit exceeded,790,26.38
2,Depth limit exceeded,524,17.50
3,Teralizer exception,97,3.24
4,Execution timeout,28,0.93
5,OutOfMemoryError,16,0.53


## Test Execution Failures

Analysis of test execution failures by exception type and generalization variant.

In [7]:
# Get test execution data by variant
test_executions_by_variant_raw = get_test_executions_by_variant_data(conn_dev)
test_failures_by_variant = compute_test_failures_by_variant_summary(
    test_executions_by_variant_raw
)

print("Test execution failures by variant:")
display(
    test_failures_by_variant[
        ["variant", "total_executions", "null_pct", "too_many_filter_pct", "other_pct"]
    ]
)

# Generate LaTeX table
test_failures_by_variant_table = generate_test_failures_by_variant_table(
    test_failures_by_variant
)
print("\n=== Test Failures by Variant Table ===\n")
print(test_failures_by_variant_table)

# Save LaTeX table
save_latex_table(test_failures_by_variant_table, "tab-exclusions-test-fails-by-variant")

# Generate and save CSV data
test_failures_by_variant_csv = generate_test_failures_by_variant_csv(
    test_failures_by_variant
)
csv_path = save_csv_data(
    test_failures_by_variant_csv,
    "exclusions-test-fails-by-variant-data",
    "Test execution failures by variant",
)

print(f"\nTest failures by variant data exported to: {csv_path}")
print(f"Shape: {test_failures_by_variant_csv.shape}")
print("Sample data:")
display(test_failures_by_variant_csv.head())

Test execution failures by variant:


,variant,total_executions,null_pct,too_many_filter_pct,other_pct
0,ORIGINAL,39014,99.661660,0.000000,0.338340
1,INITIAL,19306,100.000000,0.000000,0.000000
2,BASELINE,33142,99.933619,0.000000,0.066381
3,NAIVE_10_TRIES,33110,90.755059,6.744186,2.500755
4,NAIVE_50_TRIES,33110,88.402295,8.873452,2.724252
5,NAIVE_200_TRIES,33110,88.151616,9.075808,2.772576
6,IMPROVED_10_TRIES,33110,93.911205,3.591060,2.497735
7,IMPROVED_50_TRIES,33110,93.524615,3.693748,2.781637
8,IMPROVED_200_TRIES,33110,93.334340,3.820598,2.845062



=== Test Failures by Variant Table ===

\begin{table}[H]
  \caption{Test execution failure analysis by variant.}
  \label{tab:exclusions-test-fails-by-variant}
  \begin{tabular}{lrrrr}
    \toprule
    Variant & Total & \multicolumn{1}{c}{No Error} & \multicolumn{1}{c}{TooManyFilterMisses} & \multicolumn{1}{c}{Inaccurate Specification} \\
    \midrule
    \VariantOriginal{} & 39,014 & 38,882\; (99.7\%) & 0\; (\phantom{0}0.0\%) & 132\; (\phantom{0}0.3\%) \\
    \VariantInitial{} & 19,306 & 19,306\; (100.0\%) & 0\; (\phantom{0}0.0\%) & 0\; (\phantom{0}0.0\%) \\
    \VariantBaseline{} & 33,142 & 33,120\; (99.9\%) & 0\; (\phantom{0}0.0\%) & 22\; (\phantom{0}0.1\%) \\
    \midrule
    \VariantNaiveA{} & 33,110 & 30,049\; (90.8\%) & 2,233\; (\phantom{0}6.7\%) & 828\; (\phantom{0}2.5\%) \\
    \VariantNaiveB{} & 33,110 & 29,270\; (88.4\%) & 2,938\; (\phantom{0}8.9\%) & 902\; (\phantom{0}2.7\%) \\
    \VariantNaiveC{} & 33,110 & 29,187\; (88.2\%) & 3,005\; (\phantom{0}9.1\%) & 918\; (\phantom

,variant,total_executions,null_count,null_percentage,too_many_filter_count,too_many_filter_percentage,other_count,other_percentage
0,\VariantOriginal{},39014,38882,99.661660,0,0.000000,132,0.338340
1,\VariantInitial{},19306,19306,100.000000,0,0.000000,0,0.000000
2,\VariantBaseline{},33142,33120,99.933619,0,0.000000,22,0.066381
3,\VariantNaiveA{},33110,30049,90.755059,2233,6.744186,828,2.500755
4,\VariantNaiveB{},33110,29270,88.402295,2938,8.873452,902,2.724252


## Test Execution Failures by Projects

Analysis of test execution failures grouped by projects using test generalization, showing failure type distribution across projects.

In [8]:
# Get test failures data by projects
test_failures_by_projects_raw = get_test_failures_by_projects_data(conn_dev)
test_failures_by_projects = compute_test_failures_by_projects_summary(
    test_failures_by_projects_raw
)

print("Test execution failures by project (projects using test generalization):")
display(
    test_failures_by_projects[
        [
            "project_name",
            "total_executions",
            "null_pct",
            "too_many_filter_pct",
            "other_pct",
        ]
    ]
)

# Generate LaTeX table
test_failures_by_projects_table = generate_test_failures_by_projects_table(
    test_failures_by_projects
)
print("\n=== Test Failures by Projects Table ===\n")
print(test_failures_by_projects_table)

# Save LaTeX table
save_latex_table(
    test_failures_by_projects_table, "tab-exclusions-test-fails-by-project"
)

# Generate and save CSV data
test_failures_by_projects_csv = generate_test_failures_by_projects_csv(
    test_failures_by_projects
)
csv_path = save_csv_data(
    test_failures_by_projects_csv,
    "exclusions-test-fails-by-project-data",
    "Test execution failures by project (projects using test generalization)",
)

print(f"\nTest failures by projects data exported to: {csv_path}")
print(f"Shape: {test_failures_by_projects_csv.shape}")
print("Sample data:")
display(test_failures_by_projects_csv.head())

Test execution failures by project (projects using test generalization):


,project_name,total_executions,null_pct,too_many_filter_pct,other_pct
4,eqbench-es-default-1s,57137,96.256366,3.335842,0.407792
5,eqbench-es-default-10s,59033,95.951417,3.638643,0.409940
3,eqbench-es-default-60s,60231,95.764639,3.783766,0.451595
2,commons-utils-es-default-1s,27598,91.017465,4.985869,3.996666
6,commons-utils-es-default-10s,30968,90.067166,5.625161,4.307672
1,commons-utils-es-default-60s,31572,89.192956,6.002154,4.804890
0,commons-utils,18367,93.139870,2.760385,4.099744



=== Test Failures by Projects Table ===

\begin{table}[H]
  \caption{Test execution failure analysis by project.}
  \label{tab:exclusions-test-fails-by-project}
  \begin{tabular}{lrrrr}
    \toprule
    Project & Total & \multicolumn{1}{c}{No Error} & \multicolumn{1}{c}{TooManyFilterMisses} & \multicolumn{1}{c}{Inaccurate Specification} \\
    \midrule
    \DatasetEqBenchA{} & 57,137 & 54,998\; (96.3\%) & 1,906\; (\phantom{0}3.3\%) & 233\; (\phantom{0}0.4\%) \\
    \DatasetEqBenchB{} & 59,033 & 56,643\; (96.0\%) & 2,148\; (\phantom{0}3.6\%) & 242\; (\phantom{0}0.4\%) \\
    \DatasetEqBenchC{} & 60,231 & 57,680\; (95.8\%) & 2,279\; (\phantom{0}3.8\%) & 272\; (\phantom{0}0.5\%) \\
    \midrule
    \DatasetCommonsA{} & 27,598 & 25,119\; (91.0\%) & 1,376\; (\phantom{0}5.0\%) & 1,103\; (\phantom{0}4.0\%) \\
    \DatasetCommonsB{} & 30,968 & 27,892\; (90.1\%) & 1,742\; (\phantom{0}5.6\%) & 1,334\; (\phantom{0}4.3\%) \\
    \DatasetCommonsC{} & 31,572 & 28,160\; (89.2\%) & 1,895\; (\phantom{

,project_name,total_executions,null_count,null_percentage,too_many_filter_count,too_many_filter_percentage,other_count,other_percentage
0,eqbench-es-default-1s,57137,54998,96.256366,1906,3.335842,233,0.407792
1,eqbench-es-default-10s,59033,56643,95.951417,2148,3.638643,242,0.409940
2,eqbench-es-default-60s,60231,57680,95.764639,2279,3.783766,272,0.451595
3,commons-utils-es-default-1s,27598,25119,91.017465,1376,4.985869,1103,3.996666
4,commons-utils-es-default-10s,30968,27892,90.067166,1742,5.625161,1334,4.307672


## Summary

This notebook has analyzed the limitations of the test generalization approach across multiple dimensions:

1. **Overall Exclusions**: Shows included and excluded counts by variant and level
2. **Exclusion Breakdown**: Separates exclusions into proactive filtering and reactive failures
3. **Filtering Results**: Details which specific filters cause exclusions and their effectiveness
4. **SPF Failures**: Categorizes symbolic execution failures by error type
5. **Test Failures**: Analyzes runtime test execution failures by variant and by project

All results have been exported as both LaTeX tables and CSV data for further analysis.

## RQ4 Analysis Complete

Generated outputs:
- `tab-exclusions-summary.tex` - Overall exclusions by variant and level
- `exclusions-summary-data.csv` - Overall exclusions data
- `tab-exclusions-breakdown.tex` - Exclusion breakdown (filtering vs failures)
- `exclusions-breakdown-data.csv` - Exclusion breakdown data
- `tab-exclusions-breakdown-extended.tex` - Extended dataset exclusion breakdown
- `exclusions-breakdown-extended-data.csv` - Extended dataset breakdown data
- `tab-exclusions-filtering.tex` - Filtering results by filter type and variant
- `exclusions-filtering-data.csv` - Filtering results data
- `tab-exclusions-filtering-extended.tex` - Extended dataset filtering results
- `exclusions-filtering-extended-data.csv` - Extended dataset filtering data
- `tab-exclusions-spf.tex` - SPF execution failures by error type
- `exclusions-spf-data.csv` - SPF execution failures data
- `tab-exclusions-test-fails-by-variant.tex` - Test execution failures by variant
- `exclusions-test-fails-by-variant-data.csv` - Test execution failures by variant data
- `tab-exclusions-test-fails-by-project.tex` - Test execution failures by project
- `exclusions-test-fails-by-project-data.csv` - Test execution failures by project data